## Instalação de Dependências

In [ ]:
!pip install yfinance

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf

from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2


## Obtenção e Análise dos Dados

Nesta seção realizamos o download dos dados históricos da ação via `yfinance`, inspecionamos a estrutura do DataFrame e justificamos a escolha do **`RobustScaler`** para a normalização das features.


In [ ]:
# Especifique o símbolo da empresa e o período desejado
symbol = 'NVDA'
start_date = '2018-01-01'
end_date   = '2026-06-20'

# Download dos dados históricos
df = yf.download(symbol, start=start_date, end=end_date)
df


### Refatorando Colunas do DataFrame

O `yfinance` retorna um MultiIndex (tipo de preço × ticker). Vamos simplificar para um único nível de colunas e converter o índice `Date` em coluna regular.


In [ ]:
# Simplificar MultiIndex → nomes principais
df.columns = df.columns.get_level_values(0)

# Converter o índice 'Date' em coluna regular
df.reset_index(inplace=True)

print('DataFrame com colunas refatoradas:')
display(df.head())


### Análise das Colunas do DataFrame

O DataFrame contém dados históricos de preços da ação **NVDA** obtidos via `yfinance`. Abaixo, a descrição de cada coluna e seu uso potencial no modelo:

| Coluna | Descrição | Uso no Modelo |
|--------|-----------|---------------|
| **`Close`** | Preço de fechamento (ajustado) | Target principal do LSTM |
| **`High`** | Máxima do dia | Feature de volatilidade |
| **`Low`** | Mínima do dia | Feature de volatilidade |
| **`Open`** | Preço de abertura | Feature de tendência intraday |
| **`Volume`** | Quantidade de ações negociadas | Feature de liquidez/convicção |


### Série Temporal do Preço de Fechamento

In [ ]:
sns.set_style('whitegrid')

plt.figure(figsize=(12, 6))
plt.plot(df['Date'], df['Close'], label='Preço de Fechamento')
plt.title(f'Série Temporal do Preço de Fechamento de {symbol}')
plt.xlabel('Data')
plt.ylabel('Preço de Fechamento')
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### Distribuição e Estatísticas Descritivas

In [ ]:
numerical_cols = ['Close', 'High', 'Low', 'Open', 'Volume']
df_numerical = df[numerical_cols]

print('Estatísticas Descritivas das Colunas Numéricas:')
display(df_numerical.describe())

# Histogramas
plt.figure(figsize=(15, 10))
for i, col in enumerate(numerical_cols):
    plt.subplot(2, 3, i + 1)
    sns.histplot(df_numerical[col], kde=True)
    plt.title(f'Distribuição de {col}')
    plt.xlabel(col)
    plt.ylabel('Frequência')
plt.tight_layout()
plt.show()


### Boxplots — Visualização de Outliers

In [ ]:
plt.figure(figsize=(15, 10))
for i, col in enumerate(numerical_cols):
    plt.subplot(2, 3, i + 1)
    sns.boxplot(y=df[col])
    plt.title(f'Boxplot de {col}')
    plt.ylabel(col)
    plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


### Análise Quantitativa de Outliers — `Close`

In [ ]:
print('Quartis para os Preços de Fechamento:')
print(df['Close'].describe()[['25%', '50%', '75%']])

Q1 = df['Close'].quantile(0.25)
Q3 = df['Close'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df['Close'][(df['Close'] < lower_bound) | (df['Close'] > upper_bound)]

print('\nOutliers nos Preços de Fechamento:')
if not outliers.empty:
    display(outliers)
else:
    print('Nenhum outlier encontrado.')

total_data_points = len(df['Close'])
num_outliers = len(outliers)
if total_data_points > 0:
    percentage_outliers = (num_outliers / total_data_points) * 100
    print(f'\nPorcentagem de outliers nos Preços de Fechamento: {percentage_outliers:.2f}%')


### Justificativa para o Uso do `RobustScaler`

A análise exploratória acima evidencia dois padrões relevantes nos dados:

1. **Assimetria à direita (skewness):** os preços (`Close`, `High`, `Low`, `Open`) e o `Volume` apresentam distribuições com cauda longa para valores altos, conforme visualizado nos histogramas.
2. **Outliers expressivos:** os boxplots e a análise IQR confirmam a presença de pontos fora dos limites esperados, especialmente em `Volume` e nos preços em períodos de alta volatilidade.

Diante disso, o **`RobustScaler`** é a escolha mais adequada entre as alternativas comuns:

| Scaler | Baseia-se em | Sensível a Outliers? | Indicado quando |
|--------|-------------|----------------------|-----------------|
| `MinMaxScaler` | Min / Max | ✅ Muito | Dados sem outliers, intervalo fixo necessário |
| `StandardScaler` | Média / Desvio-padrão | ✅ Sim | Distribuição aproximadamente normal |
| **`RobustScaler`** | **Mediana / IQR** | **❌ Pouco** | **Dados com outliers ou assimetria** |

O `RobustScaler` centraliza os dados na **mediana** e escala pelo **Intervalo Interquartil (IQR)** — estatísticas resistentes a valores extremos. Isso garante que os outliers característicos de dados financeiros não distorçam a escala da maioria dos dados, permitindo que o modelo LSTM aprenda padrões mais robustos e generalizáveis.


## Pré-processamento e Engenharia de Features


### Feature Engineering — Indicadores Técnicos

Além das colunas brutas do OHLCV, adicionamos indicadores técnicos amplamente usados em análise financeira. Eles capturam **tendência**, **momentum** e **volatilidade**, enriquecendo o espaço de entrada do modelo:

| Feature | Descrição |
|---------|----------|
| `Return_1d` | Retorno diário (%) |
| `MA_7` / `MA_21` | Médias móveis de 7 e 21 dias |
| `Ratio_MA7_MA21` | Razão entre as MAs (sinal de cruzamento) |
| `Volatility_7d` | Desvio-padrão dos retornos (7 dias) |
| `RSI_14` | Índice de Força Relativa (14 dias) |
| `MACD` / `MACD_Signal` | Moving Average Convergence Divergence |
| `BB_Upper` / `BB_Lower` | Bandas de Bollinger (20 dias, 2σ) |
| `Volume_MA_7` | Média móvel do volume (7 dias) |
| `Volume_Ratio` | Volume relativo à sua média |


In [ ]:
# ── Feature Engineering ───────────────────────────────────────────

# Retorno diário
df['Return_1d'] = df['Close'].pct_change()

# Médias móveis
df['MA_7']  = df['Close'].rolling(7).mean()
df['MA_21'] = df['Close'].rolling(21).mean()
df['Ratio_MA7_MA21'] = df['MA_7'] / df['MA_21']

# Volatilidade realizada (7 dias)
df['Volatility_7d'] = df['Return_1d'].rolling(7).std()

# RSI (14 dias)
delta = df['Close'].diff()
gain  = delta.clip(lower=0).rolling(14).mean()
loss  = (-delta.clip(upper=0)).rolling(14).mean()
df['RSI_14'] = 100 - (100 / (1 + gain / loss))

# MACD
ema12 = df['Close'].ewm(span=12, adjust=False).mean()
ema26 = df['Close'].ewm(span=26, adjust=False).mean()
df['MACD']        = ema12 - ema26
df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()

# Bandas de Bollinger (20 dias)
bb_ma  = df['Close'].rolling(20).mean()
bb_std = df['Close'].rolling(20).std()
df['BB_Upper'] = bb_ma + 2 * bb_std
df['BB_Lower'] = bb_ma - 2 * bb_std

# Features de volume
df['Volume_MA_7']  = df['Volume'].rolling(7).mean()
df['Volume_Ratio'] = df['Volume'] / df['Volume_MA_7']

# Remover linhas com NaN gerados pelos rolling windows
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Shape após feature engineering: {df.shape}')
display(df.head())


In [ ]:
df

### Seleção de Features e Normalização

O `RobustScaler` é aplicado **apenas nos dados de treino** (`fit_transform`) e depois reutilizado nos dados de teste (`transform`), evitando *data leakage*.


In [ ]:
FEATURE_COLS = [
    'Close', 'High', 'Low', 'Open', 'Volume',
    'Return_1d', 'MA_7', 'MA_21', 'Ratio_MA7_MA21',
    'Volatility_7d', 'RSI_14', 'MACD', 'MACD_Signal',
    'BB_Upper', 'BB_Lower', 'Volume_MA_7', 'Volume_Ratio'
]

data = df[FEATURE_COLS].values
num_features = data.shape[1]
print(f'Total de features: {num_features}')

# Split temporal ANTES de normalizar (sem data leakage)
LOOK_BACK  = 60
train_size = int(len(data) * 0.7)
val_size   = int(len(data) * 0.15)

train_raw = data[:train_size]
val_raw   = data[train_size:train_size + val_size]
test_raw  = data[train_size + val_size:]

# Fit SOMENTE no treino
scaler = RobustScaler()
train_scaled = scaler.fit_transform(train_raw)
val_scaled   = scaler.transform(val_raw)
test_scaled  = scaler.transform(test_raw)

print(f'Treino: {len(train_scaled)} amostras')
print(f'Val:    {len(val_scaled)} amostras')
print(f'Teste:  {len(test_scaled)} amostras')


In [ ]:
def create_sequences(data, look_back):
    X, y = [], []
    for i in range(len(data) - look_back):
        X.append(data[i : i + look_back, :])  # todas as features
        y.append(data[i + look_back, 0])       # target: Close (índice 0)
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_scaled, LOOK_BACK)
X_val,   y_val   = create_sequences(val_scaled,   LOOK_BACK)
X_test,  y_test  = create_sequences(test_scaled,  LOOK_BACK)

print(f'X_train: {X_train.shape}  |  y_train: {y_train.shape}')
print(f'X_val:   {X_val.shape}    |  y_val:   {y_val.shape}')
print(f'X_test:  {X_test.shape}   |  y_test:  {y_test.shape}')


## Validação de Hiperparâmetros com TimeSeriesSplit

O K-Fold convencional **não pode ser usado** em séries temporais pois embaralha os dados, permitindo que o modelo treine com informações do futuro (*data leakage*). O `TimeSeriesSplit` resolve isso garantindo que cada fold de validação seja **sempre posterior** ao treino:

```
Fold 1: [████ treino ████]────────────────── [▓ val ▓]
Fold 2: [████████ treino ████████]────────── [▓ val ▓]
Fold 3: [████████████ treino ████████████]── [▓ val ▓]
```

Usamos 5 folds para comparar combinações de hiperparâmetros. Os melhores valores encontrados aqui alimentam o treinamento final.


In [ ]:
from sklearn.model_selection import TimeSeriesSplit
import itertools
import pandas as pd


### Grade de Hiperparâmetros

Avaliamos três dimensões:

- **`look_back`**: janela temporal de entrada (dias que o modelo 'olha para trás')
- **`units`**: número de unidades nas camadas LSTM
- **`dropout`**: taxa de regularização Dropout


In [ ]:
PARAM_GRID = {
    'look_back': [30, 60, 90],
    'units':     [32, 64],
    'dropout':   [0.2, 0.3],
}

combinations = list(itertools.product(
    PARAM_GRID['look_back'],
    PARAM_GRID['units'],
    PARAM_GRID['dropout']
))

print(f'Total de combinações a avaliar: {len(combinations)}')
for c in combinations:
    print(f'  look_back={c[0]:3d} | units={c[1]:3d} | dropout={c[2]}')


In [ ]:
def build_model_search(look_back, num_features, units, dropout_rate):
    """Versão compacta do modelo para busca de hiperparâmetros."""
    m = Sequential([
        Bidirectional(
            LSTM(units, return_sequences=True, kernel_regularizer=l2(1e-4)),
            input_shape=(look_back, num_features)
        ),
        BatchNormalization(),
        Dropout(dropout_rate),
        LSTM(units // 2, return_sequences=False, kernel_regularizer=l2(1e-4)),
        BatchNormalization(),
        Dropout(dropout_rate),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    m.compile(optimizer=Adam(learning_rate=1e-3), loss='huber', metrics=['mae'])
    return m


### Busca com TimeSeriesSplit (5 folds)

> **Nota:** Esta célula treina 12 combinações × 5 folds com early stopping leve. Pode levar alguns minutos dependendo do hardware.


In [ ]:
N_SPLITS      = 5
SEARCH_EPOCHS = 30
tscv = TimeSeriesSplit(n_splits=N_SPLITS)

es_search = EarlyStopping(monitor='val_loss', patience=5,
                           restore_best_weights=True, verbose=0)

results = []

for look_back_s, units_s, dropout_s in combinations:
    fold_losses = []

    for fold, (train_idx, val_idx) in enumerate(tscv.split(data)):
        raw_train = data[train_idx]
        raw_val   = data[val_idx]

        # Normalização sem leakage: fit apenas no treino do fold
        sc_fold  = RobustScaler()
        s_train  = sc_fold.fit_transform(raw_train)
        s_val    = sc_fold.transform(raw_val)

        Xf_tr,  yf_tr  = create_sequences(s_train, look_back_s)
        Xf_val, yf_val = create_sequences(s_val,   look_back_s)

        if len(Xf_tr) == 0 or len(Xf_val) == 0:
            continue

        m = build_model_search(look_back_s, num_features, units_s, dropout_s)
        m.fit(
            Xf_tr, yf_tr,
            epochs=SEARCH_EPOCHS,
            batch_size=32,
            validation_data=(Xf_val, yf_val),
            callbacks=[es_search],
            verbose=0
        )
        fold_losses.append(min(m.history.history['val_loss']))
        tf.keras.backend.clear_session()

    if fold_losses:
        mean_loss = np.mean(fold_losses)
        std_loss  = np.std(fold_losses)
        results.append({
            'look_back':      look_back_s,
            'units':          units_s,
            'dropout':        dropout_s,
            'mean_val_loss':  mean_loss,
            'std_val_loss':   std_loss,
        })
        print(f'look_back={look_back_s:3d} | units={units_s:3d} | dropout={dropout_s} '
              f'-> val_loss={mean_loss:.5f} +/- {std_loss:.5f}')

print('\nBusca concluida.')


In [ ]:
results_df = pd.DataFrame(results).sort_values('mean_val_loss').reset_index(drop=True)
print('Ranking de hiperparametros (menor val_loss = melhor):')
display(results_df)

best = results_df.iloc[0]
BEST_LOOK_BACK = int(best['look_back'])
BEST_UNITS     = int(best['units'])
BEST_DROPOUT   = float(best['dropout'])

print(f'\nMelhores hiperparametros:')
print(f'  look_back = {BEST_LOOK_BACK}')
print(f'  units     = {BEST_UNITS}')
print(f'  dropout   = {BEST_DROPOUT}')


In [ ]:
# Heatmap: mean_val_loss por look_back x units (melhor dropout)
best_dropout_val = results_df.groupby('dropout')['mean_val_loss'].mean().idxmin()
pivot = (
    results_df[results_df['dropout'] == best_dropout_val]
    .pivot(index='look_back', columns='units', values='mean_val_loss')
)

plt.figure(figsize=(7, 4))
sns.heatmap(pivot, annot=True, fmt='.5f', cmap='YlOrRd_r', linewidths=0.5)
plt.title(f'Val Loss por Look-back x Units (dropout={best_dropout_val})')
plt.tight_layout()
plt.show()

# Barplot com desvio padrao entre folds
fig, ax = plt.subplots(figsize=(12, 4))
labels = [
    f'lb={r.look_back}\nu={r.units}\nd={r.dropout}'
    for _, r in results_df.iterrows()
]
ax.bar(labels, results_df['mean_val_loss'],
       yerr=results_df['std_val_loss'], capsize=4,
       color='steelblue', alpha=0.7)
ax.set_ylabel('Val Loss (Huber)')
ax.set_title('Comparacao de Hiperparametros — mean +/- std across folds')
ax.tick_params(axis='x', labelsize=8)
plt.tight_layout()
plt.show()


### Atualização do Split com os Melhores Hiperparâmetros

Com o `look_back` ótimo definido pela busca, recriamos as sequências de treino, validação e teste antes do treinamento final.


In [ ]:
LOOK_BACK = BEST_LOOK_BACK

X_train, y_train = create_sequences(train_scaled, LOOK_BACK)
X_val,   y_val   = create_sequences(val_scaled,   LOOK_BACK)
X_test,  y_test  = create_sequences(test_scaled,  LOOK_BACK)

print(f'LOOK_BACK final: {LOOK_BACK}')
print(f'X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}')


## Treinamento Robusto do Modelo LSTM

### Arquitetura — Bidirectional LSTM com Regularização

O modelo original usava dois LSTMs unidirecionais simples, sem regularização além do Dropout. A nova arquitetura incorpora:

| Melhoria | Justificativa |
|----------|---------------|
| **Bidirectional LSTM** | Aprende padrões em ambas as direções temporais |
| **BatchNormalization** | Estabiliza e acelera o treinamento |
| **Regularização L2** | Penaliza pesos grandes, reduzindo overfitting |
| **EarlyStopping** | Para o treino quando val_loss para de melhorar |
| **ReduceLROnPlateau** | Reduz a taxa de aprendizado em platôs |
| **Split 70/15/15** | Conjunto de validação independente do teste |
| **17 features** | OHLCV + indicadores técnicos (RSI, MACD, Bollinger…) |


In [ ]:
def build_model(look_back, num_features, units, dropout_rate):
    model = Sequential([
        Bidirectional(
            LSTM(units, return_sequences=True, kernel_regularizer=l2(1e-4)),
            input_shape=(look_back, num_features)
        ),
        BatchNormalization(),
        Dropout(dropout_rate),
        LSTM(units // 2, return_sequences=True, kernel_regularizer=l2(1e-4)),
        BatchNormalization(),
        Dropout(dropout_rate),
        LSTM(units // 4, return_sequences=False),
        BatchNormalization(),
        Dropout(dropout_rate * 0.5),
        Dense(32, activation='relu', kernel_regularizer=l2(1e-4)),
        Dense(1)
    ])
    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss='huber',
        metrics=['mae']
    )
    return model

# Usar os melhores hiperparâmetros encontrados pelo TimeSeriesSplit
model = build_model(LOOK_BACK, num_features, BEST_UNITS, BEST_DROPOUT)
model.summary()


### Callbacks de Treinamento

- **EarlyStopping** (`patience=15`): interrompe se a `val_loss` não melhorar por 15 épocas e restaura os melhores pesos automaticamente.
- **ReduceLROnPlateau** (`patience=7`): divide o learning rate por 2 em platôs, permitindo convergência mais fina.
- **ModelCheckpoint**: salva o melhor modelo em disco.


In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        'best_lstm_model.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=0
    )
]

history = model.fit(
    X_train, y_train,
    epochs=150,            # EarlyStopping decide quando parar
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=1
)

print(f'\nTreinamento encerrado na época: {len(history.history["loss"])}')
print(f'Melhor val_loss: {min(history.history["val_loss"]):.6f}')


### Curvas de Aprendizado

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'],     label='Treino')
axes[0].plot(history.history['val_loss'], label='Validação')
axes[0].set_title('Loss (Huber)')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history['mae'],     label='Treino')
axes[1].plot(history.history['val_mae'], label='Validação')
axes[1].set_title('MAE')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('MAE (escala normalizada)')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()


## Avaliação do Modelo

In [ ]:
# Previsões no conjunto de teste
predictions = model.predict(X_test)

# Inverter a normalização — reconstruir array com num_features colunas
def inverse_close(arr_1d, scaler, num_features):
    dummy = np.zeros((len(arr_1d), num_features))
    dummy[:, 0] = arr_1d
    return scaler.inverse_transform(dummy)[:, 0]

y_pred_orig = inverse_close(predictions.flatten(), scaler, num_features)
y_test_orig = inverse_close(y_test,                scaler, num_features)

# Métricas
mae  = mean_absolute_error(y_test_orig, y_pred_orig)
mse  = mean_squared_error(y_test_orig, y_pred_orig)
rmse = np.sqrt(mse)
mape = np.mean(np.abs((y_test_orig - y_pred_orig) / y_test_orig)) * 100
# Directional Accuracy: acerta a direção do movimento?
dir_acc = np.mean(
    np.sign(np.diff(y_test_orig)) == np.sign(np.diff(y_pred_orig))
) * 100

print('=' * 50)
print('  Avaliação do Modelo (Escala Original — USD)')
print('=' * 50)
print(f'  MAE   (Erro Médio Absoluto):           ${mae:.4f}')
print(f'  RMSE  (Raiz do Erro Quadrático Médio): ${rmse:.4f}')
print(f'  MAPE  (Erro Percentual Médio):          {mape:.2f}%')
print(f'  Dir.  (Acurácia Direcional):            {dir_acc:.2f}%')
print('=' * 50)


In [ ]:
# Visualização: Real vs Previsão
plt.figure(figsize=(14, 6))
plt.plot(y_test_orig,  label='Preço Real',         color='steelblue', alpha=0.8)
plt.plot(y_pred_orig,  label='Previsão do Modelo', color='tomato', linestyle='--')
plt.title(f'Real vs Previsão — {symbol} (Conjunto de Teste)')
plt.xlabel('Dias')
plt.ylabel('Preço de Fechamento (USD)')
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

# Scatter: Real vs Previsto
plt.figure(figsize=(6, 6))
plt.scatter(y_test_orig, y_pred_orig, alpha=0.4, color='steelblue', s=10)
lims = [min(y_test_orig.min(), y_pred_orig.min()),
        max(y_test_orig.max(), y_pred_orig.max())]
plt.plot(lims, lims, 'r--', lw=1.5, label='Predição Perfeita')
plt.xlabel('Preço Real')
plt.ylabel('Preço Previsto')
plt.title('Scatter — Real vs Previsto')
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()
